<a href="https://colab.research.google.com/github/DraconMagnus/IA-Aplicada/blob/main/Carlos_Klein_Copy_of_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [3]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [4]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1

!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [5]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess

import time

import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):

    try:

        requests.get("http://localhost:11434")

        print("Servidor de Ollama listo.")

        break

    except requests.exceptions.ConnectionError:

        time.sleep(1)

else:

    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.6


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [6]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)
MODELO = "llama3.1:8b"

!ollama pull {MODELO}

!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [7]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")

print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [8]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": prompt_zero_shot}]

)

print("Zero-shot:", response_zero.message.content)

Zero-shot: Positivo.


In [10]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."

Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."

Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."

Sentimiento:"""

response_few = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": prompt_few_shot}]

)

print("Few-shot:", response_few.message.content)

# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: El sentimiento de la reseña es Mixto.


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [11]:
# Razonamiento paso a paso (chain-of-thought)
problema = (

    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "

    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "

    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."

)

response_cot = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": problema}]

)

print(response_cot.message.content)

Analicemos el problema paso a paso:

1. El primer tren sale de la ciudad A a 80 km/h y viaja durante 2 horas. En 2 horas, la distancia recorrida por el primer tren es:
Distancia = Velocidad x Tiempo
= 80 km/h x 2 horas
= 160 km

Entonces, el primer tren se encuentra a 160 km de la ciudad A.

2. El segundo tren sale de la ciudad A a 120 km/h. Como queremos saber cuánto tiempo tarda en alcanzar al primer tren, supongamos que el tiempo necesario para alcanzar al primer tren es t horas. Entonces, la distancia recorrida por el segundo tren en ese tiempo es:
Distancia = Velocidad x Tiempo
= 120 km/h x t horas
= 120t km

3. Como el segundo tren debe cubrir la distancia entre la ciudad A y el primer tren para alcanzarlo, podemos plantear la ecuación:
120t = 160 + 120t (ya que el segundo tren viaja a 120 km/h y el primer tren ha viajado durante t horas más de 2 horas)

Ahora, resolvamos la ecuación para t:
120t = 160 + 120t
120t - 120t = 160
0 = 160 (este es un error en mi parte. La ecuación no

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [12]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (

    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "

    "2026? Respuesta muy breve y corta."

)

response_alucinacion = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": prompt_desconocido}]

)

print(response_alucinacion.message.content)

Lo siento, pero no tengo información sobre el resultado de la final del hackathon interno de DEV.F del 14 de agosto de 2026.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [13]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer

import numpy as np

In [14]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [

    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "

    "empaque original.",

    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "

    "envío de regreso.",

    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "

    "cambio de talla."

]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [15]:
embeddings_documentos

array([[-0.01169081,  0.10756346, -0.16554247, ..., -0.22189358,
         0.43975028,  0.16927984],
       [-0.09546103, -0.32028627,  0.08932842, ..., -0.02563629,
         0.06188476, -0.17114003],
       [-0.16867578, -0.06994932,  0.02509686, ..., -0.26751643,
         0.39653844,  0.0300588 ]], dtype=float32)

In [16]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):

    embedding_pregunta = modelo_embeddings.encode([pregunta])

    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()

    indice_mas_similar = np.argmax(similitudes)

    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"

fragmento = buscar_fragmento(pregunta)

print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [17]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": prompt_rag}]

)

print(response_rag.message.content)

Lo siento, no puedes devolver algo que compraste en oferta.


## **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [3]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.11


In [4]:
# Descargar el modelo Llama e inicializar el cliente
MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


**2. Define tu base de conocimiento y genera embeddings:**

* Instala e importa la librería de  `sentence-transformers` junto con `numpy`.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings.

In [5]:
# Instalar e importar librerías
!pip install -q sentence-transformers numpy

import numpy as np
from sentence_transformers import SentenceTransformer

In [13]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# Sustituye estos ejemplos por tres fragmentos de un documento real
documentos = [
    "La CPU es el procesador de propósito general de una computadora. Ejecuta instrucciones y administra una amplia variedad de tareas,como las operaciones del sistema y los programas",
    "SLa GPU es un procesador diseñado para realizar muchas operaciones en paralelo. Se utiliza para procesar gráficos y también para tareas como el entrenamiento y la ejecución de modelos de inteligencia artificial.",
    "Una diferencia entre CPU y GPU es su diseño: la CPU suele tener menos núcleos, pero más versátiles; la GPU suele tener muchos núcleos especializados para trabajar en paralelo. Ambas se complementan"
]

# Convertir cada fragmento en una representación numérica
embeddings = modelo_embeddings.encode(documentos)

# Verificar el resultado
print("Número de documentos:", len(documentos))
print("Forma de los embeddings:", embeddings.shape)
print("Embedding del primer documento:", embeddings[0])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Número de documentos: 3
Forma de los embeddings: (3, 384)
Embedding del primer documento: [ 0.02176431  0.01326866  0.0529373   0.04841701 -0.25008187 -0.42833832
  0.14507781  0.05946493  0.1637768  -0.09619296  0.09299163 -0.24375932
 -0.0123016  -0.5398877  -0.09317177 -0.26434594  0.4532698  -0.0258154
  0.21507722 -0.39192832  0.00452532 -0.18022986 -0.17149623  0.0034622
 -0.4398113   0.16554302 -0.17447083  0.03021363 -0.10449174 -0.19350559
  0.02198298  0.04099916  0.4557461   0.20245181 -0.16483805  0.00641425
 -0.12294432 -0.02412695 -0.08682787 -0.17444015 -0.08137502  0.16779254
  0.28847066  0.0070834   0.22042786 -0.01199804 -0.161596   -0.12197084
 -0.09874853 -0.4325198  -0.28709024 -0.06363291  0.19498263 -0.16786678
  0.28393978  0.20496942  0.7805042  -0.11958835 -0.07177544 -0.00978804
  0.2378904  -0.28033948  0.220836   -0.20943825  0.26900747 -0.19058006
  0.30073124  0.21009538  0.02904968  0.18607226 -0.2983158  -0.05798499
  0.05404861  0.07142934 -0.05958888

**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [14]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    # Convertir la pregunta en un embedding
    embedding_pregunta = modelo_embeddings.encode(pregunta)

    # Calcular la similitud coseno con cada documento
    similitudes = (
        np.dot(embeddings, embedding_pregunta)
        / (np.linalg.norm(embeddings, axis=1)
           * np.linalg.norm(embedding_pregunta))
    )

    # Encontrar y devolver el fragmento más parecido
    indice = np.argmax(similitudes)
    return documentos[indice]

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [15]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
pregunta = "Cuáles son las diferencias entre CPU y GPU?"

respuesta = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": MODELO,
        "prompt": pregunta,
        "stream": False
    },
    timeout=120
)

respuesta.raise_for_status()
respuesta_sin_rag = respuesta.json()["response"]

print("RESPUESTA SIN RAG:")
print(respuesta_sin_rag)

RESPUESTA SIN RAG:
Una CPU (Unidad central de procesamiento) y una GPU (Unidad de procesamiento gráfico) son componentes distintos de un sistema informático. La CPU se encarga de procesar la información general de un ordenador, incluyendo tareas de escritorio, navegación por Internet, aplicaciones de oficina y más. Por otro lado, la GPU está diseñada específicamente para procesar información gráfica de alta velocidad, como renderizado 3D, video juegos, y computación en paralelo.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [16]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
# Buscar el fragmento más relacionado con la pregunta
fragmento = buscar_fragmento(pregunta)

# Enviar a Llama la pregunta y el fragmento recuperado
prompt = f"""
Responde la pregunta usando únicamente el contexto siguiente.
Si el contexto no contiene la respuesta, indica que no lo sabes.

Contexto:
{fragmento}

Pregunta:
{pregunta}
"""

respuesta = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": MODELO,
        "prompt": prompt,
        "stream": False
    },
    timeout=120
)

respuesta.raise_for_status()
respuesta_con_rag = respuesta.json()["response"]

print("FRAGMENTO RECUPERADO:")
print(fragmento)

print("\nRESPUESTA CON RAG:")
print(respuesta_con_rag)

FRAGMENTO RECUPERADO:
Una diferencia entre CPU y GPU es su diseño: la CPU suele tener menos núcleos, pero más versátiles; la GPU suele tener muchos núcleos especializados para trabajar en paralelo. Ambas se complementan

RESPUESTA CON RAG:
Las diferencias entre CPU y GPU son:

- El número de núcleos: la CPU suele tener menos núcleos pero más versátiles, mientras que la GPU tiene muchos núcleos especializados para trabajar en paralelo.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [17]:
# Mostrar ambas respuestas para comparar
print("PREGUNTA:")
print(pregunta)

print("\n--- RESPUESTA SIN RAG ---")
print(respuesta_sin_rag)

print("\n--- RESPUESTA CON RAG ---")
print(respuesta_con_rag)

print("\n--- FRAGMENTO DEL DOCUMENTO ---")
print(fragmento)

PREGUNTA:
Cuáles son las diferencias entre CPU y GPU?

--- RESPUESTA SIN RAG ---
Una CPU (Unidad central de procesamiento) y una GPU (Unidad de procesamiento gráfico) son componentes distintos de un sistema informático. La CPU se encarga de procesar la información general de un ordenador, incluyendo tareas de escritorio, navegación por Internet, aplicaciones de oficina y más. Por otro lado, la GPU está diseñada específicamente para procesar información gráfica de alta velocidad, como renderizado 3D, video juegos, y computación en paralelo.

--- RESPUESTA CON RAG ---
Las diferencias entre CPU y GPU son:

- El número de núcleos: la CPU suele tener menos núcleos pero más versátiles, mientras que la GPU tiene muchos núcleos especializados para trabajar en paralelo.

--- FRAGMENTO DEL DOCUMENTO ---
Una diferencia entre CPU y GPU es su diseño: la CPU suele tener menos núcleos, pero más versátiles; la GPU suele tener muchos núcleos especializados para trabajar en paralelo. Ambas se complement